In [1]:
import os
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models


In [2]:
def get_device():
    if torch.backends.mps.is_available():
        print("✅ Using Apple MPS (Metal) GPU")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("✅ Using CUDA GPU")
        return torch.device("cuda")
    else:
        print("⚠️ Using CPU")
        return torch.device("cpu")

device = get_device()


✅ Using Apple MPS (Metal) GPU


In [13]:
APTOS_ROOT = "/Users/sabarish/Desktop/DR_DATA/APTOS-2019 dataset"

APTOS_TEST_CSV   = os.path.join(APTOS_ROOT, "test.csv")
APTOS_TEST_IMAGES = os.path.join(APTOS_ROOT, "test_images", "test_images")


MODEL_A_PATH = "/Users/sabarish/Desktop/DR_CODE/modelA_resnet18_best.pth"
MODEL_B_PATH = "/Users/sabarish/Desktop/DR_CODE/modelB_resnet18_realCAM.pth"

print("APTOS test CSV:   ", APTOS_TEST_CSV)
print("APTOS test images:", APTOS_TEST_IMAGES)


APTOS test CSV:    /Users/sabarish/Desktop/DR_DATA/APTOS-2019 dataset/test.csv
APTOS test images: /Users/sabarish/Desktop/DR_DATA/APTOS-2019 dataset/test_images/test_images


In [14]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

aptos_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])


In [15]:
class AptosDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform

        self.image_col = "id_code"
        self.label_col = "diagnosis"   # 0–4 DR grade

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx][self.image_col])
        label  = int(self.df.iloc[idx][self.label_col])

        # Most APTOS images are PNG; add .png if not present
        if not img_id.lower().endswith((".png", ".jpg", ".jpeg")):
            img_name = img_id + ".png"
        else:
            img_name = img_id

        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


In [16]:
aptos_test_dataset = AptosDataset(
    csv_path=APTOS_TEST_CSV,
    image_dir=APTOS_TEST_IMAGES,
    transform=aptos_transform,
)

aptos_test_loader = DataLoader(
    aptos_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

print("✅ APTOS test samples:", len(aptos_test_dataset))


✅ APTOS test samples: 366


In [17]:
def build_model(num_classes: int = 5):
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


In [18]:
def load_model(ckpt_path):
    model = build_model(num_classes=5)
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    print(f"✅ Loaded model: {ckpt_path}")
    return model


In [19]:
model_a = load_model(MODEL_A_PATH)
model_b = load_model(MODEL_B_PATH)


✅ Loaded model: /Users/sabarish/Desktop/DR_CODE/modelA_resnet18_best.pth
✅ Loaded model: /Users/sabarish/Desktop/DR_CODE/modelB_resnet18_realCAM.pth


In [20]:
def evaluate_model(model, dataloader):
    criterion = nn.CrossEntropyLoss()

    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


In [21]:
print("\n🧪 Evaluating on APTOS-2019 test set...\n")

loss_a, acc_a = evaluate_model(model_a, aptos_test_loader)
loss_b, acc_b = evaluate_model(model_b, aptos_test_loader)

print("=========== APTOS-2019 TEST RESULTS ===========")
print(f"Model A  | Loss: {loss_a:.4f} | Accuracy: {acc_a:.4f}")
print(f"Model B  | Loss: {loss_b:.4f} | Accuracy: {acc_b:.4f}")
print("===============================================")



🧪 Evaluating on APTOS-2019 test set...

=========== APTOS-2019 TEST RESULTS ===========
Model A  | Loss: 0.3828 | Accuracy: 0.8579
Model B  | Loss: 0.3384 | Accuracy: 0.8770
